# Practical 7: YOLO (YOLOv8-cls) Classification on the Flowers Dataset

**Aim:** To classify flower images into 5 classes with a YOLOv8 classification model (yolov8n-cls) fine-tuned from ImageNet weights, and to understand why classification (not detection) is used for this dataset.

> Reconstructed code (the source PDF has no code). Not executed by the author - run cells in order (Kernel -> Restart & Run All) and check dataset paths in the first cells. Expected values: Top-1 accuracy ~95%, Top-5 = 100% (only 5 classes).; Flowers dataset has NO bounding boxes -> use yolov8n-cls (classification head), not detection.

## Cell 1 - Install and import

**What it does:**
- Installs and imports Ultralytics YOLO plus helper libraries. `IMG_EXT` lists accepted image file types.
- The first line starting with `!` is a Jupyter shell command (remove it if ultralytics is already installed).

*Expected output (illustrative):* Installation messages, then no output.

In [ ]:
!pip install -q ultralytics

import random
import shutil
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from ultralytics import YOLO

IMG_EXT = {".jpg", ".jpeg", ".png"}

## Cell 2 - Download the flowers dataset (skip if you already have it)

**What it does:**
- Downloads the 5-class flowers archive if the folder is not present. If your lab has its own flowers folder, set `SRC = Path('your/folder')` (one sub-folder per class).
- Prints each class name and how many images it contains.

*Expected output (illustrative):* Classes: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips'] with counts 633, 898, 641, 699, 799.

In [ ]:
SRC = Path("flower_photos")

if not SRC.exists():
    url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
    urllib.request.urlretrieve(url, "flower_photos.tgz")
    with tarfile.open("flower_photos.tgz") as tar:
        tar.extractall(".")

classes = sorted(d.name for d in SRC.iterdir() if d.is_dir())
print("Classes:", classes)
for c in classes:
    print(c, len([f for f in (SRC / c).iterdir() if f.suffix.lower() in IMG_EXT]))

## Cell 3 - Create the train / val folder structure (80/20)

**What it does:**
- YOLO classification expects `train/` and `val/` folders, each with one sub-folder per class.
- Each class is shuffled (fixed seed) and 20% of its images go to `val`, the rest to `train` - so every class appears in both sets.
- Files are copied (originals stay untouched).

*Expected output (illustrative):* train ~2,939 images
val ~731 images

In [ ]:
DST = Path("flowers_split")
random.seed(42)

for class_dir in sorted(d for d in SRC.iterdir() if d.is_dir()):
    files = sorted(f for f in class_dir.iterdir() if f.suffix.lower() in IMG_EXT)
    random.shuffle(files)
    n_val = int(0.2 * len(files))
    for split, subset in (("val", files[:n_val]), ("train", files[n_val:])):
        out_dir = DST / split / class_dir.name
        out_dir.mkdir(parents=True, exist_ok=True)
        for f in subset:
            shutil.copy(f, out_dir / f.name)

for split in ("train", "val"):
    total = sum(1 for _ in (DST / split).rglob("*") if _.suffix.lower() in IMG_EXT)
    print(split, total, "images")

## Cell 4 - Train YOLOv8-cls (fine-tuning from ImageNet weights)

**What it does:**
- `yolov8n-cls.pt` = YOLOv8 **nano** classification model, pre-trained on ImageNet (downloaded automatically).
- `data` is the folder that contains `train/` and `val/`. `imgsz=224` is the input size, `batch=32` images per step, `epochs=10` passes over the data (epochs/batch are reconstructed values).
- Ultralytics chooses the optimizer automatically (AdamW/SGD) and applies built-in augmentation. Weights (`best.pt`, `last.pt`), `results.csv`, graphs and a confusion matrix are saved in `save_dir`.

*Expected output (illustrative):* Per-epoch table with columns: epoch, GPU_mem, loss, instances, Size; then validation 'top1_acc  top5_acc'. Final top-1 about 0.95, top-5 = 1.0.

In [ ]:
model = YOLO("yolov8n-cls.pt")

results = model.train(
    data=str(DST.resolve()),
    epochs=10,
    imgsz=224,
    batch=32,
)

save_dir = Path(model.trainer.save_dir)
print("Results saved in:", save_dir)

## Cell 5 - Plot the training curves from results.csv

**What it does:**
- Ultralytics logs every epoch in `results.csv`. Column names have extra spaces, so `str.strip()` cleans them.
- Left: loss going down. Right: Top-1 / Top-5 accuracy going up.

*Expected output (illustrative):* Loss curves decreasing; Top-1 accuracy climbing to ~0.9+, Top-5 flat at 1.0.

In [ ]:
df = pd.read_csv(save_dir / "results.csv")
df.columns = df.columns.str.strip()
print(df.columns.tolist())

plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(df["epoch"], df["train/loss"], label="Train loss")
plt.plot(df["epoch"], df["val/loss"], label="Validation loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(df["epoch"], df["metrics/accuracy_top1"], label="Top-1 accuracy")
plt.plot(df["epoch"], df["metrics/accuracy_top5"], label="Top-5 accuracy")
plt.title("Validation accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## Cell 6 - Validate the best model (Top-1 / Top-5)

**What it does:**
- Loads the best checkpoint (highest validation accuracy) and evaluates it on `val/`.
- `top1` is the normal accuracy; `top5` is 1.0 because there are only 5 classes.

*Expected output (illustrative):* Top-1 accuracy ~0.95, Top-5 accuracy = 1.0 (expected from your PDF).

In [ ]:
best = YOLO(str(save_dir / "weights" / "best.pt"))
VAL_DIR = DST / "val"

metrics = best.val(data=str(DST.resolve()), imgsz=224)
print("Top-1 accuracy:", round(metrics.top1, 4))
print("Top-5 accuracy:", round(metrics.top5, 4))

## Cell 7 - Confusion matrix and precision / recall / F1

**What it does:**
- Predicts every validation image and compares with the true folder label.
- `confusion_matrix` shows which flowers are confused (e.g. roses <-> tulips); `classification_report` gives per-class precision, recall and F1.

*Expected output (illustrative):* A 5 x 5 confusion matrix with a strong diagonal and a report table with per-class values around 0.93-0.97.

In [ ]:
CM_SIZE = 6
INCLUDE_VALUES = True

names = best.names
name_to_id = {v: k for k, v in names.items()}

val_files, y_true = [], []
for class_dir in sorted(VAL_DIR.iterdir()):
    if class_dir.is_dir():
        for f in sorted(class_dir.iterdir()):
            if f.suffix.lower() in IMG_EXT:
                val_files.append(str(f))
                y_true.append(name_to_id[class_dir.name])

y_pred = []
for start in range(0, len(val_files), 64):
    batch_results = best.predict(val_files[start:start + 64], imgsz=224, verbose=False)
    y_pred.extend(int(r.probs.top1) for r in batch_results)

labels = list(range(len(names)))
label_names = [names[i] for i in labels]

cm = confusion_matrix(y_true, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(CM_SIZE, CM_SIZE))
ConfusionMatrixDisplay(cm, display_labels=label_names).plot(
    ax=ax, cmap="Blues", xticks_rotation=90, include_values=INCLUDE_VALUES, colorbar=False)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, labels=labels, target_names=label_names, digits=3))

## Cell 8 - Predict a single image

**What it does:**
- `r.probs.top1` is the index of the most probable class and `top1conf` its probability.
- Shows the image with its predicted class.

*Expected output (illustrative):* Example: 'Predicted: daisy  confidence: 0.98 / True: daisy' with the picture.

In [ ]:
sample = val_files[0]
r = best.predict(sample, imgsz=224, verbose=False)[0]

pred_name = r.names[r.probs.top1]
print("Predicted:", pred_name, " confidence:", round(float(r.probs.top1conf), 3))
print("True     :", Path(sample).parent.name)

plt.imshow(Image.open(sample))
plt.title(pred_name)
plt.axis("off")
plt.show()

## Result / Conclusion

A YOLOv8n-cls model fine-tuned from ImageNet weights classified 5 flower types with ~95% Top-1 accuracy; classification was the correct task because the dataset has no bounding boxes.